# Protocolo MQTT na Automação Industrial e a Stack MING

## Prática de Comunicação IIoT: Publicação e Subscrição com HiveMQ e Python

Este notebook aborda os fundamentos teóricos e a implementação prática do protocolo **MQTT (Message Queuing Telemetry Transport)**, demonstrando como integrar o chão de fábrica à **Stack MING (MQTT, InfluxDB, Node-RED e Grafana)** para monitoramento e observabilidade industrial em tempo real.

### Objetivos de Aprendizagem
- Compreender a teoria do protocolo MQTT, padrão Publish/Subscribe, hierarquia de tópicos e níveis de QoS (0, 1 e 2).
- Dominar a arquitetura da **Stack MING** (MQTT + InfluxDB + Node-RED + Grafana) na Indústria 4.0.
- Desenvolver clientes **Publisher (Publicador)** e **Subscriber (Assinante)** em Python utilizando a biblioteca `paho-mqtt` conectados ao broker público **HiveMQ**.
- Simular a ingestão de telemetria industrial de motores, sensores e atuadores com payloads JSON e conversão para Influx Line Protocol.

---


## 1. Fundamentação Teórica: O Protocolo MQTT

### 1.1 O que é o MQTT?
O **MQTT (Message Queuing Telemetry Transport)** é um protocolo de comunicação leve e assíncrono baseado na camada de transporte **TCP/IP** (porta padrão `1883` sem TLS, ou `8883` com TLS). Foi criado em 1999 por Andy Stanford-Clark (IBM) e Arlen Nipper (Eurotech) com o objetivo inicial de monitorar oleodutos via satélite com largura de banda restrita e alta latência.

Hoje, padronizado pela **OASIS** e **ISO (ISO/IEC 20922)**, o MQTT é o protocolo mais adotado na **Internet das Coisas Industrial (IIoT)** e em arquiteturas de Indústria 4.0.

### 1.2 O Padrão Publish / Subscribe (Pub/Sub)
Diferente do modelo tradicional cliente-servidor HTTP (Request/Response), o MQTT adota o modelo **Publish/Subscribe**, que proporciona **triplo desacoplamento**:
1. **Desacoplamento Espacial:** Publicadores e assinantes não precisam saber o endereço IP ou a identidade uns dos outros; apenas conhecem o Broker.
2. **Desacoplamento Temporal:** Publicador e assinante não precisam estar conectados no mesmo instante para trocar informações (suportado por mensagens retidas e sessões persistentes).
3. **Desacoplamento de Sincronização:** As operações de envio e recebimento de dados ocorrem de forma assíncrona, sem bloquear os nós da rede industrial.

![Teoria e Fundamentos do Protocolo MQTT](img/teoria_mqtt_pubsub.png)

---

### 1.3 Elementos Fundamentais do MQTT

| Elemento | Descrição e Papel na Rede |
| :--- | :--- |
| **Broker (Servidor Central)** | O ponto focal da arquitetura. Recebe todas as mensagens, filtra por tópico e entrega para os assinantes autorizados. Exemplo: **HiveMQ**, Eclipse Mosquitto, EMQX. |
| **Publisher (Publicador)** | Dispositivo ou software que envia dados (telemetria de CLP, sensores, inversores) sob um determinado **Tópico**. |
| **Subscriber (Assinante)** | Aplicação que se inscreve em um ou mais tópicos para consumir mensagens em tempo real (SCADA, Node-RED, InfluxDB, scripts analíticos). |
| **Tópico (Topic)** | String com estrutura hierárquica separada por barras (`/`), usada pelo broker para rotear as mensagens. Ex: `fabrica/linha1/prensa01/temperatura`. |
| **Payload** | O conteúdo transmitido. O MQTT é agnóstico ao formato: pode transportar texto puro, números, JSON, Protocol Buffers ou dados binários. |

### 1.4 Hierarquia de Tópicos e Wildcards (Curingas)
Na automação de uma fábrica, os tópicos devem seguir um padrão semântico estruturado:
`<empresa>/<unidade>/<setor>/<linha>/<equipamento>/<variavel>`

Os assinantes podem utilizar dois tipos de curingas (wildcards) para monitorar múltiplos tópicos:
- **`+` (Curinga de nível único):** Substitui exatamente um nível hierárquico.
  - Exemplo: `fabrica/linha1/+/temperatura` (captura a temperatura de todos os motores/máquinas da linha 1).
- **`#` (Curinga multi-nível):** Substitui todos os níveis subsequentes (deve ser posicionado no final do tópico).
  - Exemplo: `fabrica/linha1/#` (captura todas as variáveis e eventos de toda a linha 1).

### 1.5 Níveis de Qualidade de Serviço (QoS - Quality of Service)
O MQTT permite balancear confiabilidade e consumo de largura de banda através de 3 níveis de QoS:
1. **QoS 0 — At most once (No máximo uma vez / "Fogo e Esqueça"):**
   - A mensagem é enviada sem confirmação de entrega (*PUBACK*). É o método mais rápido e leve. Ideal para leituras periódicas frequentes onde a perda ocasional de uma amostra não afeta o processo.
2. **QoS 1 — At least once (Pelo menos uma vez):**
   - O broker confirma o recebimento com *PUBACK*. Caso ocorra falha de rede antes da confirmação, o publicador retransmite. Garante a entrega, porém pode gerar duplicatas que a aplicação receptora deve tratar.
3. **QoS 2 — Exactly once (Exatamente uma vez):**
   - Utiliza um handshake em 4 vias (*PUBLISH* → *PUBREC* → *PUBREL* → *PUBCOMP*). Garante que a mensagem seja entregue exatamente uma única vez, sem perdas nem duplicações. Indicado para transações críticas, faturamento ou acionamentos de segurança.

### 1.6 Recursos Avançados do MQTT
- **Retained Messages (Mensagens Retidas):** O broker armazena a última mensagem com a flag `retain=True`. Quando um novo cliente se inscreve no tópico, ele recebe imediatamente o último estado conhecido sem precisar aguardar uma nova publicação.
- **Last Will and Testament (LWT):** Testamento configurado na conexão. Se o cliente sofrer uma desconexão abrupta (falha de hardware/energia), o Broker publica automaticamente uma mensagem de alerta no tópico de status (ex: `"status": "OFFLINE"`).
- **Keep Alive e Ping:** Mecanismo periódico de *PINGREQ* / *PINGRESP* para manter a conexão TCP ativa e detectar quedas silenciosas de rede.

---


## 2. A Stack MING na Automação Industrial

A **Stack MING** é uma das arquiteturas de referência mais consagradas na Indústria 4.0 para conectar o universo de **Tecnologia da Automação (TA/OT)** ao universo de **Tecnologia da Informação (TI/IT)**.

![Arquitetura da Stack MING na Automação Industrial](img/stack_ming_automacao.png)

### 2.1 Componentes da Stack MING

```
┌──────────────────┐      ┌─────────────┐      ┌────────────┐      ┌───────────┐      ┌───────────┐
│ SENSORES / CLP   │ ──── │ MQTT Broker │ ──── │  Node-RED  │ ──── │ InfluxDB  │ ──── │  Grafana  │
│ (Chão de Fábrica)│      │  (HiveMQ)   │      │ (ETL/Flow) │      │  (TSDB)   │      │(Dashboard)│
└──────────────────┘      └─────────────┘      └────────────┘      └───────────┘      └───────────┘
     Nível 0/1              M - MQTT             N - Node-RED        I - InfluxDB       G - Grafana
```

1. **M — MQTT (Message Broker - HiveMQ / Mosquitto):**
   - Funciona como a espinha dorsal de comunicação assíncrona. Desacopla os transmissores de dados dos consumidores.
   - Permite que milhares de tags industriais sejam publicadas com baixíssimo consumo de rede.

2. **I — InfluxDB (Time Series Database):**
   - Banco de dados otimizado especificamente para séries temporais (telemetria indexada no tempo).
   - Grava milhões de pontos por segundo em formato colunar com compressão e políticas de retenção automática de dados (*Retention Policies*).
   - Utiliza a estrutura *Line Protocol*: `<measurement>,<tag_set> <field_set> <timestamp_ns>`.

3. **N — Node-RED (Pipeline ETL e Orquestração de Fluxos):**
   - Ferramenta de desenvolvimento baseada em fluxo (*low-code*) para integração de hardware e serviços.
   - Subscreve tópicos MQTT, valida esquemas JSON, aplica regras de negócio, filtra ruídos e formata os dados para o formato aceito pelo InfluxDB ou aciona atuadores de volta ao CLP.

4. **G — Grafana (Plataforma de Observabilidade e Visualização):**
   - Conecta-se ao InfluxDB para consultar métricas via linguagem Flux ou InfluxQL.
   - Exibe painéis interativos de gestão à vista: gráficos de linha, termômetros, manômetros, cálculo de OEE (*Overall Equipment Effectiveness*) e gestão de alarmes preditivos.

---


## 3. Prática: Instalação e Preparação do Ambiente Python

Utilizaremos a biblioteca oficial **`paho-mqtt`** do projeto Eclipse. Ela suporta conexões com brokers MQTT locais (Mosquitto) e em nuvem (como o broker público do **HiveMQ** em `broker.hivemq.com`).

In [ ]:
# Instalação da biblioteca paho-mqtt (caso ainda não esteja instalada)
!pip install paho-mqtt -q

### Configuração dos Parâmetros do Broker HiveMQ
O HiveMQ disponibiliza um broker público para testes comunitários e acadêmicos:
- **Host:** `broker.hivemq.com`
- **Porta TCP Padrão:** `1883` (não criptografada) / `8883` (com TLS)
- **WebSockets:** `8000` (porta HTTP) / `8884` (porta HTTPS)

> **Dica de Boas Práticas:** Como o broker público é compartilhado globalmente, use um prefixo exclusivo para os tópicos do seu projeto (por exemplo, `senai/turma_aut/aluno_xyz/...`) para evitar colisões com outros usuários.

In [ ]:
import paho.mqtt.client as mqtt
import json
import time
import random
import uuid

# Parâmetros de Conexão com o HiveMQ
BROKER_HOST = "broker.hivemq.com"
BROKER_PORT = 1883
KEEP_ALIVE = 60

# Prefixo exclusivo para evitar conflito no broker público
BASE_TOPIC = "senai/automacao_n1/linha_usinagem"

print(f"Configuração definida:")
print(f"• Broker: {BROKER_HOST}:{BROKER_PORT}")
print(f"• Tópico Base: {BASE_TOPIC}/#")


---

## 4. Implementação do Assinante (Subscriber)

O cliente **Subscriber** se conecta ao broker HiveMQ, subscreve os tópicos de telemetria e trata as mensagens recebidas através de funções de callback (*on_connect*, *on_message*).

O método `client.loop_start()` cria uma thread em segundo plano que mantém o loop de rede ativo, processando mensagens sem travar a execução do notebook.

In [ ]:
# Buffer para armazenar as mensagens recebidas pelo Subscriber no notebook
historico_mensagens_recebidas = []

# 1. Callback executado quando o cliente se conecta ao Broker
def on_connect_subscriber(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print("[SUBSCRIBER] Conexão estabelecida com sucesso ao Broker HiveMQ!")
        # Subscrevendo com wildcard '#' para ouvir todas as variáveis da linha
        topico_assinatura = f"{BASE_TOPIC}/#"
        client.subscribe(topico_assinatura, qos=1)
        print(f"[SUBSCRIBER] Assinatura ativa no tópico: '{topico_assinatura}' (QoS 1)")
    else:
        print(f"[SUBSCRIBER] Falha na conexão. Código de retorno: {rc}")

# 2. Callback executado quando uma mensagem chega em qualquer tópico assinado
def on_message_subscriber(client, userdata, msg):
    try:
        payload_str = msg.payload.decode('utf-8')
        dados_json = json.loads(payload_str)
        
        # Armazenar no histórico
        historico_mensagens_recebidas.append({
            "topico": msg.topic,
            "qos": msg.qos,
            "dados": dados_json,
            "timestamp_recebido": time.time()
        })
        
        print(f"\n[SUBSCRIBER RECEBEU] Tópico: '{msg.topic}' | QoS: {msg.qos}")
        print(f"  → Payload JSON: {json.dumps(dados_json, indent=2)}")
    except Exception as e:
        print(f"[SUBSCRIBER ERRO] Não foi possível decodificar mensagem no tópico {msg.topic}: {e}")

# Criação do cliente Subscriber (compatível com paho-mqtt v1 e v2)
subscriber_id = f"sub_smart_n1_{uuid.uuid4().hex[:6]}"
try:
    # Paho-MQTT v2
    client_sub = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=subscriber_id)
except AttributeError:
    # Fallback Paho-MQTT v1
    client_sub = mqtt.Client(client_id=subscriber_id)

client_sub.on_connect = on_connect_subscriber
client_sub.on_message = on_message_subscriber

# Conexão e início do loop assíncrono em background
print(f"Iniciando Subscriber (ID: {subscriber_id})...")
client_sub.connect(BROKER_HOST, BROKER_PORT, keepalive=KEEP_ALIVE)
client_sub.loop_start()
time.sleep(1) # Breve pausa para confirmação do handshake


---

## 5. Implementação do Publicador (Publisher — Simulação de Sensor Industrial)

O cliente **Publisher** simula um CLP / Edge Gateway de chão de fábrica coletando grandezas físicas de uma célula robotizada de usinagem:
- Temperatura do Mancal do Motor (`°C`)
- Velocidade de Vibração RMS (`mm/s`)
- Pressão Hidráulica do Sistema (`bar`)
- Corrente Elétrica do Spindle (`A`)
- Status Operacional (`"RODANDO"`, `"ALERTA"`, `"EM_FALHA"`)

In [ ]:
# Criação do cliente Publisher
publisher_id = f"pub_clp_n1_{uuid.uuid4().hex[:6]}"
try:
    client_pub = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=publisher_id)
except AttributeError:
    client_pub = mqtt.Client(client_id=publisher_id)

def on_connect_publisher(client, userdata, flags, rc, properties=None):
    if rc == 0:
        print(f"[PUBLISHER] CLP conectado com sucesso ao HiveMQ (ID: {publisher_id})")
    else:
        print(f"[PUBLISHER] Erro na conexão: {rc}")

client_pub.on_connect = on_connect_publisher
client_pub.connect(BROKER_HOST, BROKER_PORT, keepalive=KEEP_ALIVE)
client_pub.loop_start()
time.sleep(1)


### 5.1 Envio de Telemetria Contínua para o Broker
Vamos disparar uma sequência de 5 ciclos de telemetria simulando o comportamento de 2 motores da linha de usinagem.

In [ ]:
print("=== INICIANDO PUBLICAÇÃO DE TELEMETRIA INDUSTRIAL NO HIVEMQ ===\n")

motores = ["torno_cnc_01", "fresa_cnc_02"]

for ciclo in range(1, 6):
    print(f"--- CICLO DE SCAN CLP #{ciclo} ---")
    
    for motor in motores:
        # Simulação de leituras de sensores analógicos do motor
        temperatura = round(random.uniform(45.0, 78.0), 2)
        vibracao = round(random.uniform(0.8, 4.5), 2)
        pressao = round(random.uniform(140.0, 160.0), 1)
        corrente = round(random.uniform(12.0, 24.5), 2)
        
        status = "EM_FALHA" if temperatura > 75.0 or vibracao > 4.0 else ("ALERTA" if temperatura > 68.0 else "OPERANDO_NORMAL")
        
        # Estrutura padrão de payload JSON para Indústria 4.0
        payload = {
            "equipamento_id": motor,
            "planta": "Fabrica_SP",
            "setor": "Usinagem_N1",
            "ciclo": ciclo,
            "metricas": {
                "temperatura_mancal_c": temperatura,
                "vibracao_rms_mms": vibracao,
                "pressao_hidraulica_bar": pressao,
                "corrente_spindle_a": corrente
            },
            "status_operacao": status,
            "timestamp_utc": time.time()
        }
        
        # Tópico específico de acordo com a hierarquia semântica
        topico_publicacao = f"{BASE_TOPIC}/{motor}/telemetria"
        mensagem_json = json.dumps(payload)
        
        # Publicação via MQTT com QoS 1
        info = client_pub.publish(topico_publicacao, mensagem_json, qos=1)
        info.wait_for_publish(timeout=2.0)
        print(f"[PUBLISHER ENVIOU] Tópico: {topico_publicacao} | Status: {status}")
        
    # Intervalo entre os ciclos de varredura
    time.sleep(1.5)

print("\nPublicações concluídas! Aguardando o Subscriber capturar todas as mensagens...")
time.sleep(2)


---

## 6. Pipeline MING: Conversão para Node-RED e InfluxDB Line Protocol

Nesta etapa, demonstramos exatamente o que o nó de função do **Node-RED** executa ao receber o JSON do MQTT e convertê-lo no **Line Protocol** do InfluxDB para persistência em séries temporais:

**Sintaxe do Line Protocol do InfluxDB:**
```text
<measurement>,<tag_set> <field_set> <timestamp_ns>
```
- **Measurement:** Nome da tabela/métrica (ex: `telemetria_motores`).
- **Tags (Indexadas para buscas rápidas):** `equipamento=torno_cnc_01,setor=Usinagem_N1,planta=Fabrica_SP`.
- **Fields (Valores numéricos e strings):** `temperatura=62.4,vibracao=1.8,pressao=145.2,status="OPERANDO_NORMAL"`.
- **Timestamp:** Tempo em nanosegundos (ou milissegundos).

In [ ]:
def simular_etl_nodered_para_influx(msg_mqtt):
    """
    Simula o processamento ETL de uma Function Node do Node-RED:
    Entrada: Objeto JSON recebido via MQTT
    Saída: String formatada em InfluxDB Line Protocol
    """
    dados = msg_mqtt["dados"]
    eq_id = dados["equipamento_id"]
    setor = dados["setor"]
    planta = dados["planta"]
    status = dados["status_operacao"]
    
    m = dados["metricas"]
    temp = m["temperatura_mancal_c"]
    vib = m["vibracao_rms_mms"]
    press = m["pressao_hidraulica_bar"]
    corr = m["corrente_spindle_a"]
    
    # Timestamp em nanosegundos (padrão InfluxDB)
    ts_ns = int(dados["timestamp_utc"] * 1_000_000_000)
    
    # Formatação do Line Protocol
    line_protocol = (
        f"telemetria_usinagem,"
        f"equipamento={eq_id},setor={setor},planta={planta} "
        f"temperatura={temp},vibracao={vib},pressao={press},corrente={corr},status=\"{status}\" "
        f"{ts_ns}"
    )
    return line_protocol

print("=== SIMULAÇÃO DA TRANSFORMAÇÃO NODE-RED → INFLUXDB LINE PROTOCOL ===\n")
for i, item in enumerate(historico_mensagens_recebidas[:4], 1):
    linha_influx = simular_etl_nodered_para_influx(item)
    print(f"Registro {i}:")
    print(f"  → Influx Line Protocol: {linha_influx}\n")


---

## 7. Encerramento Seguro das Conexões MQTT

Ao finalizar a sessão de testes, é fundamental parar as threads de rede (`loop_stop()`) e desconectar os clientes (`disconnect()`) para liberar os recursos no broker.

In [ ]:
# Finalização dos clientes MQTT
print("Encerrando conexões com o Broker HiveMQ...")
client_pub.loop_stop()
client_pub.disconnect()

client_sub.loop_stop()
client_sub.disconnect()

print(f"Total de mensagens capturadas com sucesso pelo Subscriber: {len(historico_mensagens_recebidas)}")
print("Conexões encerradas com sucesso!")


---

## 8. Exercícios de Fixação e Avaliação

### Questão 1 (Teoria do MQTT)
Explique por que o padrão **Publish/Subscribe** do MQTT é mais eficiente para o tráfego de dados de sensores industriais em redes de baixa largura de banda do que o modelo **Request/Response** do protocolo HTTP/REST.

### Questão 2 (Qualidade de Serviço - QoS)
Em um sistema de telemetria industrial:
1. Qual nível de QoS (0, 1 ou 2) você escolheria para o envio contínuo da temperatura de um forno a cada 500 ms? Justifique.
2. Qual nível de QoS você escolheria para o envio de um comando de parada de emergência (*E-Stop*) ou acionamento de válvula crítica? Justifique.

### Questão 3 (Arquitetura da Stack MING)
Descreva o ciclo de vida completo de uma variável de processo (ex: vibração de um motor elétrico) desde a leitura física no sensor até a sua renderização em um painel do Grafana, detalhando a função específica de cada letra da sigla **MING** (**M**QTT, **I**nfluxDB, **N**ode-RED, **G**rafana).

### Questão 4 (Desafio Prático em Código)
Modifique o código do **Publisher** para adicionar o envio de uma mensagem de estado inicial com a flag `retain=True` no tópico `senai/automacao_n1/linha_usinagem/status_geral` contendo `{"estado": "SISTEMA_PRONTO"}`. Explique o que acontece quando um novo **Subscriber** se conecta após esse envio.
